# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', 'No title')}: {getattr(metadata, 'description', 'No description')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")

for rs in record_sets:
    print(f"Record Set @id: {rs.id}, name: {getattr(rs, 'name', 'N/A')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} (name: {getattr(field, 'name', 'N/A')}, type: {getattr(field, 'data_type', 'N/A')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_sets_ids = [rs.id for rs in record_sets]
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found for {record_set_id}")

# Display the first record set DataFrame's columns and head (if available)
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"DataFrame columns in {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Select your primary record set
record_set_id = main_record_set_id  # Use the main one detected earlier
df = dataframes[record_set_id]

# Choose numeric field by @id. Here, we try to pick 'gad7_score', 'phq9_score', or similar, if present
possible_numeric_fields = [c for c in df.columns if any(v in c.lower() for v in ['score', 'gad', 'phq', 'psq', 'age']) and df[c].dtype in [np.int64, np.float64, float, int]]

if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    print(f"Using numeric field: {numeric_field} (@id)")
else:
    numeric_field = df.columns[0]
    print(f"No obvious numeric field found, using: {numeric_field} (@id)")

threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 10
try:
    filtered_df = df[df[numeric_field] > threshold]
except Exception:
    filtered_df = df.copy()

print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

if np.issubdtype(filtered_df[numeric_field].dtype, np.number):
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a field. For survey data, 'gender' or 'level_of_education' might be present
group_fields = [c for c in df.columns if 'gender' in c.lower() or 'education' in c.lower() or 'village' in c.lower()]
if group_fields:
    group_field = group_fields[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field} (showing mean values for numeric columns):")
    display(grouped_df.head())
else:
    group_field = None
    print('No suitable grouping field found.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic histogram for the chosen numeric field
if np.issubdtype(df[numeric_field].dtype, np.number):
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

# Boxplot by group, if available
if group_field:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Key Observations:**

- The dataset contains mental health survey responses from Kilifi County, Kenya, structured with fields including demographic data and psychometric assessment scores.
- The main numeric fields (e.g., GAD-7, PHQ-9, PSQ scores) are suitable for analysis, filtering, and visualization.
- Grouping by demographic information can reveal patterns or disparities in mental health indicators across different groups.
- This notebook can be extended with richer analysis, such as cross-tabulations, missing value exploration, or predictive modeling based on these fields.

_Remember to consult the data documentation and respect the privacy and sensitivity of personal information contained in the dataset._